# 🧱 Part 17: Speculative Decoding: Let a Small Model Help a Large Model

> **Previous context**: KV Cache speeds a single model. Speculative decoding uses a second model to reduce expensive target-model steps.
> **Goal for this part**: Implement draft-then-verify acceptance and understand why the output distribution can remain correct.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. Core idea

A small draft model proposes several tokens quickly. The large target model verifies them in parallel.

## 1. Acceptance

If the target model agrees enough with the draft token, we accept it. If not, we correct and continue.

## 2. Why it helps

The target model still guards quality, but one target forward pass can validate multiple proposed tokens.

## 3. Limits

Speedup depends on draft quality, target cost, batch shape, and implementation overhead.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] Draft model proposes tokens.
- [ ] Target model verifies them.
- [ ] Accepted drafts reduce the number of expensive target-model steps.

Next, continue through the code cells for the Inference part and inspect the printed observations.


In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

In [2]:
# Teaching note: follow this line to see the main step.
def speculative_accept(draft_probs, target_probs, draft_tokens):
    'Read the values printed above and connect them to the concept in this cell.'
    accepted = []
    for i, (dp, tp, tok) in enumerate(zip(draft_probs, target_probs, draft_tokens)):
        # Teaching note: follow this line to see the main step.
        accept_prob = min(1.0, tp / dp) if dp > 0 else 0.0
        
        # Teaching note: follow this line to see the main step.
        if torch.rand(1).item() < accept_prob:
            accepted.append(tok)
        else:
            # Teaching note: follow this line to see the main step.
            break
    return accepted

# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
draft_tokens = [15, 23, 8, 42]
draft_probs = [0.8, 0.7, 0.6, 0.5]  # Teaching note: follow this line to see the main step.

# Teaching note: follow this line to see the main step.
target_probs = [0.9, 0.8, 0.2, 0.1]  # Teaching note: follow this line to see the main step.

print(f"Read the values printed above and connect them to the concept in this cell.{draft_tokens}")
print(f"Read the values printed above and connect them to the concept in this cell.{draft_probs}")
print(f"Read the values printed above and connect them to the concept in this cell.{target_probs}")
print()

for i in range(len(draft_tokens)):
    ratio = target_probs[i] / draft_probs[i]
    accept_prob = min(1.0, ratio)
    status = 'Read the values printed above and connect them to the concept in this cell.' if ratio >= 1 else f"🎲 {accept_prob:.0%}Read the values printed above and connect them to the concept in this cell."
    print(f"  token {draft_tokens[i]}: p_target/p_draft = {ratio:.2f} → {status}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

In [3]:
# Teaching note: follow this line to see the main step.
def simulate_speedup(K=5, accept_rate=0.6, num_tokens=100):
    'Read the values printed above and connect them to the concept in this cell.'
    target_calls = 0
    tokens_generated = 0
    
    while tokens_generated < num_tokens:
        target_calls += 1
        # Teaching note: follow this line to see the main step.
        # Teaching note: follow this line to see the main step.
        expected_accepted = min(K, int(1 / (1 - accept_rate)))
        tokens_generated += expected_accepted
    
    return num_tokens / target_calls

print('Read the values printed above and connect them to the concept in this cell.')
print()
for ar in [0.3, 0.5, 0.7, 0.9]:
    speedup = simulate_speedup(K=5, accept_rate=ar)
    print(f"Read the values printed above and connect them to the concept in this cell.{ar:.0%}Read the values printed above and connect them to the concept in this cell.{speedup:.1f}x")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

In [4]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

---

1. .2. .3. .4. .5. .6. .7. .
.